In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

df["age"] = 2023 - df["year"]
df["miles_per_year"] = df["mileage"] / df["age"].clip(lower=1)

In [2]:
features = ["manufacturer", "model", "age", "mileage", "miles_per_year",
    "drivetrain", "fuel_type", "transmission_type",
    "accidents_or_damage", "one_owner", "personal_use_only",
    "mpg_city", "mpg_hwy", "driver_rating", "driver_reviews_num",
    "seller_rating", "seller_rating_missing"]

cat_cols = ["manufacturer", "model", "drivetrain", "fuel_type", "transmission_type",
            "accidents_or_damage", "one_owner", "personal_use_only"]

df[cat_cols] = df[cat_cols].astype("category")

In [3]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
train_fit, valid = train_test_split(train, test_size=0.1, random_state=42)

In [4]:
import os, joblib
import lightgbm as lgb

os.makedirs("../models", exist_ok=True)

def train_lgbm(name, **extra):
    path = f"../models/{name}.joblib"
    if os.path.exists(path):
        print(f"Loaded {name}")
        return joblib.load(path)
    m = lgb.LGBMRegressor(n_estimators=5000, learning_rate=0.05, num_leaves=127,
                          min_child_samples=20, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=42, verbose=-1, **extra)
    m.fit(train_fit[features], np.log(train_fit["price"]),
          eval_set=[(valid[features], np.log(valid["price"]))],
          callbacks=[lgb.early_stopping(100), lgb.log_evaluation(500)])
    joblib.dump(m, path)
    return m

mid  = train_lgbm("lgbm_mid")
low  = train_lgbm("lgbm_q10", objective="quantile", alpha=0.10)
high = train_lgbm("lgbm_q90", objective="quantile", alpha=0.90)

c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[500]	valid_0's l2: 0.014723
[1000]	valid_0's l2: 0.0144629
[1500]	valid_0's l2: 0.0143468
Early stopping, best iteration is:
[1798]	valid_0's l2: 0.0143111


c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[500]	valid_0's quantile: 0.0186267
[1000]	valid_0's quantile: 0.0185217
[1500]	valid_0's quantile: 0.0185029
Early stopping, best iteration is:
[1426]	valid_0's quantile: 0.0184961


c:\Users\Manci\Desktop\GitHub projects\used-car-deal-detector\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[500]	valid_0's quantile: 0.0188461
[1000]	valid_0's quantile: 0.0187268
[1500]	valid_0's quantile: 0.0186971
Early stopping, best iteration is:
[1611]	valid_0's quantile: 0.0186928


In [5]:
X = test[features]
test = test.assign(
    pred_mid=np.exp(mid.predict(X)),
    pred_low=np.exp(low.predict(X)),
    pred_high=np.exp(high.predict(X)),
)
# Occasionally the separate models cross; make sure low <= mid <= high
test["pred_low"] = test[["pred_low", "pred_mid"]].min(axis=1)
test["pred_high"] = test[["pred_high", "pred_mid"]].max(axis=1)

In [6]:
inside = test["price"].between(test["pred_low"], test["pred_high"]).mean()
below = (test["price"] < test["pred_low"]).mean()
above = (test["price"] > test["pred_high"]).mean()
print(f"Inside range: {inside:.1%}  (target ~80%)")
print(f"Below range:  {below:.1%}  (target ~10%)")
print(f"Above range:  {above:.1%}  (target ~10%)")

test["range_width_pct"] = (test["pred_high"] - test["pred_low"]) / test["pred_mid"]
bands = pd.cut(test["price"], [0, 10_000, 20_000, 35_000, 60_000, 200_000])
test.groupby(bands, observed=True)["range_width_pct"].median().round(3)

Inside range: 75.3%  (target ~80%)
Below range:  12.3%  (target ~10%)
Above range:  12.4%  (target ~10%)


price
(0, 10000]         0.458
(10000, 20000]     0.240
(20000, 35000]     0.176
(35000, 60000]     0.154
(60000, 200000]    0.151
Name: range_width_pct, dtype: float64

In [7]:
test["discount"] = 1 - test["price"] / test["pred_mid"]
deals = (test[test["price"] < test["pred_low"]]
         .sort_values("discount", ascending=False))

cols = ["manufacturer", "model", "year", "mileage", "accidents_or_damage",
        "price", "pred_low", "pred_mid", "pred_high", "discount"]
print(f"{len(deals):,} flagged of {len(test):,} test listings")
deals[cols].head(20).round(2)

18,476 flagged of 149,822 test listings


,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
293825,GMC,Sierra 2500 Denali,2015,81258.0,No,2751.0,35409.45,35409.45,46864.92,0.92
261448,Ford,Mustang GT,1989,38359.0,No,2995.0,8822.19,18483.32,33149.60,0.84
264930,Ford,Galaxie 500 500,1968,65200.0,No,10999.0,15736.81,63811.60,85245.98,0.83
43860,BMW,328 4dr Sdn 328i RWD,2014,86768.0,Yes,2500.0,13017.07,13677.21,18661.15,0.82
334568,Hyundai,Equus Signature,2013,115818.0,No,2500.0,10447.49,13185.50,16062.00,0.81
226268,Ford,F-350 XLT Crew Cab Super Duty DRW,2003,136000.0,Unknown,3649.0,9798.50,18948.48,26065.12,0.81
598789,Porsche,Boxster Base (A5),1999,36870.0,No,3995.0,11508.58,19528.35,35498.20,0.80
539580,Mercedes-Benz,S-Class 560SEL,1986,93228.0,No,5412.0,9962.81,25807.30,38600.47,0.79
132392,Chevrolet,3500 Base,1997,128974.0,No,4000.0,11982.19,18708.13,36953.58,0.79
75174,Buick,Skylark GS 350,1968,25000.0,No,8590.0,18330.60,40168.99,61182.47,0.79


In [8]:
df["price"].value_counts().head(15)
df.loc[df["price"] == 2500, "seller_name"].value_counts().head(10)

seller_name
Car Collection Inc                   7
Suburban Ford of Sterling Heights    2
Corwin Public Wholesale              2
REM Motors LLC                       2
Central FL Motors LLC                2
Shea Buick GMC                       1
Mike Molstead Motors                 1
Starfire Auto Inc.                   1
Putnam Auto Sales Inc                1
A2Z Auto Group                       1
Name: count, dtype: int64

In [9]:
def classify(row):
    if row["year"] < 2000:
        return "Classic (out of scope)"
    if row["discount"] > 0.5:
        return "Likely data error"
    if row["accidents_or_damage"] == "Yes":
        return "Accident history"
    return "Possible deal"

deals["flag_type"] = deals.apply(classify, axis=1)
deals["flag_type"].value_counts()

flag_type
Possible deal             14071
Accident history           4134
Classic (out of scope)      161
Likely data error           110
Name: count, dtype: int64

In [10]:
real = deals[deals["flag_type"] == "Possible deal"]
real[cols].head(20).round(2)

,manufacturer,model,year,mileage,accidents_or_damage,price,pred_low,pred_mid,pred_high,discount
138282,Chevrolet,Trailblazer LT,2008,123397.0,Unknown,3999.0,5080.75,7925.06,11181.58,0.50
327002,Honda,CR-V EX,2001,129945.0,No,4499.0,5995.96,8897.67,10052.39,0.49
93945,Cadillac,Escalade Premium,2013,8300.0,Unknown,29500.0,29557.72,58338.70,84637.05,0.49
30074,BMW,Z3,2001,132000.0,Unknown,5500.0,5516.17,10870.34,12728.82,0.49
591641,Nissan,Rogue Select S,2015,86000.0,Unknown,6950.0,10580.15,13699.63,16703.16,0.49
203707,Ford,Flex SEL,2009,130074.0,No,4200.0,6859.03,8271.03,10855.77,0.49
597617,Porsche,Cayenne S,2006,178056.0,Unknown,4500.0,5833.90,8851.32,13166.14,0.49
478898,Lincoln,Town Car Signature,2004,36525.0,No,9880.0,10564.17,19400.49,19400.49,0.49
30064,BMW,Z3 2.3 Roadster,2000,120700.0,Unknown,5750.0,6832.81,11216.92,12427.84,0.49
630190,Subaru,Impreza 2.5i,2007,97771.0,No,4995.0,7408.75,9704.31,10214.64,0.49
